In [ ]:
# Prefers the real dataset at data/raw/train.parquet when present (same
# real-data schema and columns as notebooks/rebuild_pipeline.py's
# load_real_dataset()). Falls back to a synthetic generator only when the
# real file isn't there -- e.g. on a fresh clone, since data/ is gitignored.
#
# This replaces the notebook's original bug: this cell used to *always*
# fabricate synthetic "ghost data" and overwrite data/processed/base_table
# .parquet with it even when the real dataset was available, silently
# discarding real data any time the notebook was re-run top to bottom. That
# ghost data also once drew the target `y` fully independently of every
# feature (np.random.choice([0,1], p=[.56,.44])), so the trained model
# could not possibly beat a coin flip (ROC-AUC ~0.46-0.50).
#
# Canonical, up-to-date version of this logic: notebooks/rebuild_pipeline.py
# (run that script directly to regenerate outputs/ artifacts in one shot).
# This cell mirrors the same real-vs-synthetic preference so the notebook
# remains runnable standalone.
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

# Ensure directories exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../reports', exist_ok=True)

RAW_PATH = '../data/raw/train.parquet'
USE_REAL = os.path.exists(RAW_PATH)

if USE_REAL:
    print(f"Found real dataset at {RAW_PATH} -- using real data.")
    df = pd.read_parquet(RAW_PATH).sort_values('order_date').reset_index(drop=True)
    df['y'] = df['disruption_occurred'].astype(int)
    df.to_parquet('../data/processed/base_table.parquet', index=False)
    print(f"Loaded {len(df)} real rows. Positive rate: {df['y'].mean():.1%}")
    print("Data written to data/processed/base_table.parquet (from real data/raw/train.parquet)")
else:
    print(f"No real dataset at {RAW_PATH} -- falling back to synthetic data.")
    rng = np.random.default_rng(42)
    n = 2000  # Enough rows to test the pipeline
    dates = [datetime(2026, 1, 1) + timedelta(hours=x * 12) for x in range(n)]

    supplier_id = rng.choice(['S1', 'S2', 'S3'], n)
    supplier_name = pd.Series(supplier_id).map(
        {'S1': 'TechCorp', 'S2': 'GlobalSupply', 'S3': 'AeroParts'}
    ).to_numpy()
    destination_city = rng.choice(['Los Angeles', 'New York', 'Frankfurt'], n)
    transportation_mode = rng.choice(['Sea', 'Air', 'Rail'], n)
    warehouse_id = rng.choice(['WH_A', 'WH_B'], n)
    port_congestion_level_e = rng.choice(['Low', 'Medium', 'High'], n, p=[0.5, 0.3, 0.2])
    sea_x_congestion = rng.choice(['Low', 'High'], n, p=[0.7, 0.3])
    sourcing_fragility = rng.choice(['Low', 'High'], n, p=[0.7, 0.3])
    customs_clearance_hours = rng.choice(['Low', 'High'], n, p=[0.7, 0.3])
    ext_risk = rng.choice(['Low', 'High'], n, p=[0.7, 0.3])
    supplier_id_hist = rng.beta(2, 5, n)  # historical disruption propensity, skewed low
    order_value_usd = rng.uniform(5000, 85000, n).round(2)

    # Target y is a genuine (noisy) logistic function of the risk drivers,
    # not independent noise -- this is what actually gives the model
    # something to learn.
    logit = (
        -4.0
        + np.where(port_congestion_level_e == 'Medium', 1.0, 0.0)
        + np.where(port_congestion_level_e == 'High', 2.0, 0.0)
        + np.where(sourcing_fragility == 'High', 1.8, 0.0)
        + np.where(ext_risk == 'High', 1.6, 0.0)
        + np.where(sea_x_congestion == 'High', 1.0, 0.0)
        + np.where(customs_clearance_hours == 'High', 1.0, 0.0)
        + 2.2 * supplier_id_hist
        + np.where(transportation_mode == 'Sea', 0.3, np.where(transportation_mode == 'Air', -0.3, 0.0))
        + rng.normal(0, 1.0, n)  # irreducible noise so the task isn't trivially separable
    )
    p_disrupt = 1 / (1 + np.exp(-logit))
    y = rng.binomial(1, p_disrupt)

    delay_days = np.where(y == 1, rng.integers(1, 15, n), 0)
    disruption_type = np.where(
        y == 1, rng.choice(['Weather', 'Customs', 'Port Congestion', 'Supplier Failure'], n), 'None'
    )
    planned_delivery_date = [d + timedelta(days=14) for d in dates]
    actual_delivery_date = [
        pdd + timedelta(days=int(dd)) for pdd, dd in zip(planned_delivery_date, delay_days)
    ]

    df = pd.DataFrame({
        'shipment_id': [f'SC-{1000+i}' for i in range(n)],
        'order_date': dates,
        'supplier_name': supplier_name,
        'supplier_id': supplier_id,
        'planned_delivery_date': planned_delivery_date,
        'actual_delivery_date': actual_delivery_date,
        'destination_city': destination_city,
        'transportation_mode': transportation_mode,
        'order_value_usd': order_value_usd,
        'warehouse_id': warehouse_id,
        'port_congestion_level_e': port_congestion_level_e,
        'sea_x_congestion': sea_x_congestion,
        'sourcing_fragility': sourcing_fragility,
        'customs_clearance_hours': customs_clearance_hours,
        'ext_risk': ext_risk,
        'supplier_id_hist': supplier_id_hist,
        'y': y,
        'delay_days': delay_days,
        'disruption_type': disruption_type,
        'notes': 'synthetic (see notebooks/rebuild_pipeline.py)',
    }).sort_values('order_date').reset_index(drop=True)

    df.to_parquet('../data/processed/base_table.parquet', index=False)
    print(f"Generated {len(df)} synthetic rows. Positive rate: {df['y'].mean():.1%}")
    print("Data written to data/processed/base_table.parquet (SYNTHETIC FALLBACK -- no real data/raw/train.parquet found)")


In [2]:
import shap
import pandas as pd
import numpy as np
import joblib
import os

print("🧠 Initializing SHAP Explainer & Position-Mapping Drivers...")

# 1. Load the exact feature list the model was trained on
feature_path = '../outputs/features.pkl'
if not os.path.exists(feature_path):
    feature_path = 'outputs/features.pkl'

features = joblib.load(feature_path)

# 2. Load X_test from the dataset
data_path = '../data/processed/base_table.parquet'
if not os.path.exists(data_path):
    data_path = 'data/processed/base_table.parquet'
    
df = pd.read_parquet(data_path)
df = df.sort_values('order_date')
split_idx = int(len(df) * 0.8)

X_test = df.iloc[split_idx:][features].copy()

# Convert categorical text columns to numeric codes safely
for col in X_test.select_dtypes(include=['object', 'category', 'string']).columns:
    X_test[col], _ = pd.factorize(X_test[col])

X_test = X_test.select_dtypes(include=[np.number])

# 3. Safely load the model
model_path = '../outputs/model_tuned.pkl'
if not os.path.exists(model_path):
    model_path = 'outputs/model_tuned.pkl'
model = joblib.load(model_path)

# 4. Initialize SHAP Explainer
explainer = shap.Explainer(model.predict, X_test)
sample_size = min(2000, len(X_test))
X_test_sample = X_test.sample(sample_size, random_state=42)

print(f"Calculating SHAP values for {sample_size} rows...")
shap_values = explainer(X_test_sample)

# 5. Extract Top 3 Drivers per row
shap_abs = np.abs(shap_values.values)
top3_idx = np.argsort(-shap_abs, axis=1)[:, :3]

feature_names = X_test.columns
driver_df = pd.DataFrame({
    'driver_1': [feature_names[i] for i in top3_idx[:, 0]],
    'driver_2': [feature_names[i] for i in top3_idx[:, 1]],
    'driver_3': [feature_names[i] for i in top3_idx[:, 2]],
}, index=X_test_sample.index)

# 6. Load predictions dataframe
pred_path = '../outputs/predictions.csv'
if not os.path.exists(pred_path):
    pred_path = 'outputs/predictions.csv'

pred_updated = pd.read_csv(pred_path)

global_top_driver = feature_names[np.argmax(shap_abs.mean(axis=0))]
if 'driver_1' not in pred_updated.columns:
    pred_updated['driver_1'] = global_top_driver

# 🛡️ Bulletproof Positional Mapping (Bypasses Index Label Mismatches)
sample_positions = [X_test.index.get_loc(idx) for idx in X_test_sample.index]
pred_updated.iloc[sample_positions, pred_updated.columns.get_loc('driver_1')] = driver_df['driver_1'].values

# 7. Action Engine Rules Engine
RULES = {
    ('port_congestion_level_e', 'High'): 'Pre-position stock; consider air freight',
    ('sea_x_congestion', 'High'): 'Switch transport mode to Air or Rail',
    ('sourcing_fragility', 'High'): 'Dual-source this lane; qualify a backup supplier',
    ('ext_risk', 'High'): 'Increase safety stock buffer; monitor weather corridor',
    ('supplier_id_hist', 'High'): 'Escalate to procurement; request delivery confirmation',
    ('customs_clearance_hours', 'High'): 'Pre-clear documentation; engage customs broker',
}
DEFAULT = {
    'High':   'Escalate to procurement and confirm next two deliveries',
    'Medium': 'Increase check-in frequency to twice weekly',
    'Low':    'Standard monitoring — no action required',
}

if 'risk_band' in pred_updated.columns and 'driver_1' in pred_updated.columns:
    pred_updated['recommended_action'] = [
        RULES.get((d1, b), DEFAULT.get(b, 'Standard monitoring')) 
        for d1, b in zip(pred_updated['driver_1'], pred_updated['risk_band'])
    ]

# 8. Export final outputs
pred_updated.to_csv(pred_path, index=False)
print("✅ SHAP Integration Complete! outputs/predictions.csv updated with real drivers.")

Background dataset has 400 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=400 when initializing the masker.


🧠 Initializing SHAP Explainer & Position-Mapping Drivers...
Calculating SHAP values for 400 rows...


PermutationExplainer explainer: 401it [00:52,  7.25it/s]                                                                             

✅ SHAP Integration Complete! outputs/predictions.csv updated with real drivers.


In [6]:
import pandas as pd
import numpy as np
import joblib
import os
import time
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

print("Starting the 5-Model Bakeoff...")

# 🛡️ Self-healing check: If X_train doesn't exist, load and split data automatically
if 'X_train' not in locals() or 'y_train' not in locals():
    print("⚠️ 'X_train' not found in memory. Loading and splitting dataset automatically...")
    data_path = '../data/processed/base_table.parquet'
    if not os.path.exists(data_path):
        data_path = 'data/processed/base_table.parquet'
        
    df = pd.read_parquet(data_path)
    df = df.sort_values('order_date')
    split_idx = int(len(df) * 0.8)
    
    target_col = 'y' if 'y' in df.columns else df.columns[-1]
    
    # Load features if available, otherwise infer
    feature_path = '../outputs/features.pkl'
    if not os.path.exists(feature_path):
        feature_path = 'outputs/features.pkl'
        
    if os.path.exists(feature_path):
        features = joblib.load(feature_path)
    else:
        ignore_cols = ['shipment_id', 'order_date', 'supplier_name', 'planned_delivery_date', target_col]
        features = [col for col in df.columns if col not in ignore_cols]
        
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]
    
    X_train = train_df[features].copy()
    X_test = test_df[features].copy()
    
    # Factorize categorical text columns safely
    for col in X_train.select_dtypes(include=['object', 'category', 'string']).columns:
        X_train[col], _ = pd.factorize(X_train[col])
        X_test[col], _ = pd.factorize(X_test[col])
        
    X_train = X_train.select_dtypes(include=[np.number])
    X_test = X_test.select_dtypes(include=[np.number])
    
    y_train = train_df[target_col].astype(int)
    y_test = test_df[target_col].astype(int)

# 1. Define the 5 Models
models = {
    "Logistic Regression (Baseline)": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "HistGradientBoosting (Champion)": HistGradientBoostingClassifier(random_state=42)
}

results = []

# 2. Train and Evaluate Each Model
for name, m in models.items():
    start_time = time.time()
    
    # Train
    m.fit(X_train, y_train)
    
    # Predict
    y_prob = m.predict_proba(X_test)[:, 1]
    
    # Evaluate
    roc = roc_auc_score(y_test, y_prob)
    pr = average_precision_score(y_test, y_prob)
    train_time = time.time() - start_time
    
    results.append({
        "Algorithm": name,
        "ROC-AUC": round(roc, 3),
        "PR-AUC": round(pr, 3),
        "Train Time (s)": round(train_time, 2)
    })

# 3. Generate the Leaderboard
leaderboard = pd.DataFrame(results).sort_values(by="PR-AUC", ascending=False).reset_index(drop=True)
print("\n🏆 MODEL BAKEOFF RESULTS 🏆")
display(leaderboard)

Starting the 5-Model Bakeoff...

🏆 MODEL BAKEOFF RESULTS 🏆


,Algorithm,ROC-AUC,PR-AUC,Train Time (s)
0,Random Forest,0.523,0.471,0.16
1,HistGradientBoosting (Champion),0.524,0.456,0.17
2,Logistic Regression (Baseline),0.516,0.454,0.11
3,AdaBoost,0.507,0.420,0.08
4,Decision Tree,0.499,0.404,0.01


In [5]:
import time
import json
import joblib
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.ensemble import HistGradientBoostingClassifier

print("🚀 Starting Hyperparameter Tuning for HistGradientBoosting...")
start_time = time.time()

# 1. Load data and correctly define target y using delay_days from the dataset schema[cite: 1]
data_path = '../data/processed/base_table.parquet'
if not os.path.exists(data_path):
    data_path = 'data/processed/base_table.parquet'

df = pd.read_parquet(data_path)
df = df.sort_values('order_date')

# Define binary target based on actual delay data[cite: 1]
df['y'] = (df['delay_days'] > 0).astype(int)

split_idx = int(len(df) * 0.8)

# Load features list
feature_path = '../outputs/features.pkl'
if not os.path.exists(feature_path):
    feature_path = 'outputs/features.pkl'
features = joblib.load(feature_path)

X_train = df.iloc[:split_idx][features].copy()
for col in X_train.select_dtypes(include=['object', 'category', 'string']).columns:
    X_train[col], _ = pd.factorize(X_train[col])
X_train = X_train.select_dtypes(include=[np.number])
y_train = df.iloc[:split_idx]['y'].astype(int)

# 2. Setup Randomized Search with TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=3)
param_distributions = {
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_iter': [100, 200, 300],
    'max_depth': [3, 5, 7, None],
    'min_samples_leaf': [10, 20, 30]
}

search = RandomizedSearchCV(
    estimator=HistGradientBoostingClassifier(random_state=42),
    param_distributions=param_distributions,
    n_iter=15,
    scoring='roc_auc',
    cv=tscv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

# 3. Fit the Search
search.fit(X_train, y_train)

# 4. Evaluate and Save Results
elapsed_time = (time.time() - start_time) / 60
print(f"\n✅ Tuning Complete in {elapsed_time:.1f} minutes!")
print(f"Best Parameters Found:\n{json.dumps(search.best_params_, indent=2)}")

# 5. Save the Champion Model
best_model = search.best_estimator_
os.makedirs('../outputs', exist_ok=True)
joblib.dump(best_model, '../outputs/model_tuned.pkl')

print("🏆 The tuned model has been saved as 'model_tuned.pkl'.")

🚀 Starting Hyperparameter Tuning for HistGradientBoosting...
Fitting 3 folds for each of 15 candidates, totalling 45 fits

✅ Tuning Complete in 0.1 minutes!
Best Parameters Found:
{
  "min_samples_leaf": 30,
  "max_iter": 100,
  "max_depth": null,
  "learning_rate": 0.05
}
🏆 The tuned model has been saved as 'model_tuned.pkl'.


In [7]:
print("Class Distribution for 'y':")
print(df['y'].value_counts())
print("\nPercentage Breakdown:")
print(df['y'].value_counts(normalize=True) * 100)


Class Distribution for 'y':
y
1    1881
0     119
Name: count, dtype: int64

Percentage Breakdown:
y
1    94.05
0     5.95
Name: proportion, dtype: float64


In [9]:
import numpy as np
from sklearn.metrics import roc_curve, classification_report

# 1. Get predicted probabilities for disruptions
y_prob = best_model.predict_proba(X_test)[:, 1]

# 2. Use Youden's J statistic to find a balanced classification threshold
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
j_scores = tpr - fpr
optimal_threshold = thresholds[np.argmax(j_scores)]

print(f"🎯 Balanced Optimal Threshold: {optimal_threshold:.2f}")

# 3. Apply the balanced threshold
y_pred_custom = (y_prob >= optimal_threshold).astype(int)

# 4. Print the performance report (with zero_division handled cleanly)
print("\n📊 Performance Report with Balanced Threshold:")
print(classification_report(y_test, y_pred_custom, zero_division=0))

🎯 Balanced Optimal Threshold: 0.99

📊 Performance Report with Balanced Threshold:
              precision    recall  f1-score   support

           0       0.60      0.87      0.71       236
           1       0.47      0.17      0.25       164

    accuracy                           0.58       400
   macro avg       0.54      0.52      0.48       400
weighted avg       0.55      0.58      0.52       400



In [10]:
# Force a manual, practical threshold aligned with your positive rate
manual_threshold = 0.15

y_pred_custom = (y_prob >= manual_threshold).astype(int)

print(f"📊 Performance Report with Manual Threshold ({manual_threshold}):")
print(classification_report(y_test, y_pred_custom, zero_division=0))

📊 Performance Report with Manual Threshold (0.15):
              precision    recall  f1-score   support

           0       0.00      0.00      0.00       236
           1       0.41      1.00      0.58       164

    accuracy                           0.41       400
   macro avg       0.20      0.50      0.29       400
weighted avg       0.17      0.41      0.24       400



In [11]:
from imblearn.over_sampling import SMOTE
import numpy as np

# 1. Initialize SMOTE
smote = SMOTE(random_state=42)

# 2. Resample ONLY the training split
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE - Training distribution: {np.bincount(y_train)}")
print(f"After SMOTE  - Training distribution: {np.bincount(y_train_resampled)}")

# 3. Train your model on the freshly balanced data
best_model.fit(X_train_resampled, y_train_resampled)

ModuleNotFoundError: No module named 'imblearn'